# 📚 Resum de Funcions – Pràctica 3: Segmentació, Fons & Template Matching (i fonaments P2)

Aquest notebook recull totes les funcions de biblioteques externes utilitzades a les pràctiques de Visió per Computador (actualitzat per a la **Pràctica 3**, integrant i ampliant les eines de la **Pràctica 2**).

Inclou explicacions teòriques resumides, paràmetres clau, valors recomanats, tipus de dades i exemples d'ús pràctics i reproduïbles.

**Biblioteques principals:**
- `scikit-image` (`skimage`): mòduls `io`, `color`, `transform`, `exposure`, `filters`, `morphology`, `util`, `feature`, `segmentation`
- `scikit-learn` (`sklearn`): mòduls `cluster` (`AgglomerativeClustering`), `feature_extraction.image` (`grid_to_graph`)
- `scipy`: mòdul `ndimage` (`convolve`)
- `numpy` (`np`): operacions matricials, càlculs d'àlgebra, mediana temporal, padding, cerca de coordenades
- `matplotlib.pyplot` (`plt`) & `matplotlib.patches`: visualització avançada, histogrames i delimitadors rectangulars (`Rectangle`)


---
## 0. Imports generals


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

# Configuració per a visualitzacions clares
plt.rcParams['figure.dpi'] = 100
%matplotlib inline


---
## 1. `skimage.io` – Lectura i càrrega d'imatges


In [ ]:
from skimage import io

# Llegir una imatge (retorna array numpy)
# - Imatge RGB → shape (H, W, 3), dtype uint8, valors [0, 255]
# - Imatge escala de grisos → shape (H, W), dtype uint8, valors [0, 255]
# - as_gray=True → converteix automàticament a escala de grisos float64 [0.0, 1.0]

# Exemple amb imatges de la P3:
try:
    img_rgb = io.imread("images_p3/bunch-of-fresh-fruit.png")
    img_gray = io.imread("images_p3/einstein.png", as_gray=True)
    print("RGB shape:", img_rgb.shape, "| dtype:", img_rgb.dtype, "| rang:", img_rgb.min(), "-", img_rgb.max())
    print("Gray shape:", img_gray.shape, "| dtype:", img_gray.dtype, "| rang:", img_gray.min(), "-", img_gray.max())
except FileNotFoundError:
    print("Ruta d'imatges no trobada; comprova que el directori de treball contingui images_p3/")


---
## 2. `skimage.color` – Conversions de color


In [ ]:
from skimage.color import rgb2gray

# Converteix imatge RGB a escala de grisos mitjançant luminància ponderada:
# Y = 0.2125 R + 0.7154 G + 0.0721 B
# - Entrada: uint8 [0, 255] o float [0, 1] amb shape (H, W, 3)
# - Sortida: float64 [0.0, 1.0] amb shape (H, W)

if 'img_rgb' in locals():
    fruit_gray = rgb2gray(img_rgb)
    print("Convertida a grisos → shape:", fruit_gray.shape, "| dtype:", fruit_gray.dtype)


---
## 3. `skimage` & `numpy` – Conversions de tipus (`img_as_float`, `img_as_ubyte`, `astype`)


In [ ]:
from skimage import img_as_float, img_as_ubyte

# img_as_float: converteix a float64 normalitzat a [0.0, 1.0] (uint8 es divideix automàticament entre 255)
# - Imprescindible per a operacions numèriques, càlcul de gradients i mètriques SSD
test_float = img_as_float(img_rgb) if 'img_rgb' in locals() else np.zeros((10, 10), dtype=np.uint8)
print("img_as_float → dtype:", test_float.dtype, "| rang:", test_float.min(), "-", test_float.max())

# img_as_ubyte: converteix a uint8 [0, 255] multiplicant per 255 i arrodonint
# ⚠️ Atenció: si l'entrada té valors negatius o > 1.0 pot generar error o clipping!
test_ubyte = img_as_ubyte(test_float)
print("img_as_ubyte → dtype:", test_ubyte.dtype, "| rang:", test_ubyte.min(), "-", test_ubyte.max())

# astype: canvia el tipus de dades intern de numpy SENSE escalar valors
# uint8(100).astype(float) serà 100.0, NO 0.392


---
## 4. `skimage.transform` – Escala, redimensionat i rotació (`rescale`, `resize`, `rotate`)


In [ ]:
from skimage.transform import rescale, resize, rotate

# 1. rescale: canvia la mida multiplicant per una fracció
# - preserve_range=True → manté els rangs numèrics originals
# - anti_aliasing=True  → aplica filtre gaussià previ per evitar aliasing
if 'fruit_gray' in locals():
    img_downscaled = rescale(fruit_gray, 0.25, anti_aliasing=True)
    print("rescale 0.25 → original:", fruit_gray.shape, "nova:", img_downscaled.shape)

# 2. resize: força una mida exacta (H, W)
# - Retorna sempre float64 normalitzat a [0.0, 1.0]
if 'img_rgb' in locals():
    img_resized = resize(img_rgb, (128, 128), anti_aliasing=True)
    print("resize (128, 128) →", img_resized.shape, img_resized.dtype)

# 3. rotate: gira la imatge un cert angle en graus (sentit anti-horari) [NOU P3]
# - angle: graus de gir
# - resize=True: amplia el llenç perquè la imatge girada no quedi tallada als cantons
# - preserve_range=True: no normalitza el rang de valors
if 'img_gray' in locals():
    img_rot = rotate(img_gray, angle=15, resize=True)
    print("rotate 15° → original:", img_gray.shape, "girada:", img_rot.shape)


---
## 5. `skimage.exposure` & `matplotlib` – Histogrames d'intensitat


In [ ]:
from skimage.exposure import histogram

# Manera 1: skimage.exposure.histogram
# Retorna (hist, bin_centers) → útil per a càlculs numèrics o llindars
# hist, bin_centers = histogram(img_gray, nbins=64)

# Manera 2: matplotlib plt.hist
# Molt més directe per visualitzar la distribució de píxels
# .ravel() aplana la matriu 2D/3D a 1D
if 'fruit_gray' in locals():
    plt.figure(figsize=(6, 3))
    plt.hist(fruit_gray.ravel(), bins=50, color='royalblue', alpha=0.7)
    plt.title("Histograma d'intensitat")
    plt.xlabel("Valor de píxel (normalitzat)")
    plt.ylabel("Freqüència")
    plt.tight_layout()
    plt.show()


---
## 6. `scipy.ndimage` & `skimage.filters` – Convolució i filtres espacials


In [ ]:
from scipy.ndimage import convolve
from skimage.filters import gaussian, median
from skimage.morphology import disk, rectangle, diamond

# 1. Convolució manual amb convolve:
# mask normalitzada 5x5:
mask_avg = np.ones((5, 5)) / 25.0
# img_conv = convolve(img_gray, mask_avg, mode='reflect')

# 2. Filtre Gaussià (suavitzat continu):
# - sigma: desviació estàndard (a major sigma, més borrós)
# - Retorna sempre float64
# img_blur = gaussian(img_gray, sigma=2.0)

# 3. Filtre de Mediana (excel·lent per a soroll salt-and-pepper):
# - footprint: element estructurant (disk, rectangle, diamond)
# - Preserva millor les vores que el gaussià
# img_med = median(img_ubyte, disk(3))


---
## 7. `skimage.util.random_noise` – Addició de soroll sintètic


In [ ]:
from skimage.util import random_noise

# random_noise afegeix soroll i retorna SEMPRE float64 [0.0, 1.0]
# - mode='gaussian': soroll gaussià amb paràmetres 'mean' i 'var'
# - mode='s&p': soroll salt-and-pepper (impulsiu) amb paràmetre 'amount'
if 'img_gray' in locals():
    noisy_gauss = random_noise(img_gray, mode='gaussian', var=0.02)
    noisy_sp = random_noise(img_gray, mode='s&p', amount=0.05)
    print("Soroll afegit amb èxit (float64 [0.0, 1.0])")


---
## 8. `skimage.filters` & `skimage.feature` – Detecció de contorns (Sobel, Prewitt, Canny)


In [ ]:
from skimage.filters import sobel, sobel_h, sobel_v, prewitt
from skimage.feature import canny

# 1. Sobel / Prewitt: operadors diferencials basats en gradient
# - sobel_h / sobel_v: gradients parcials horitzontal (X) i vertical (Y)
# - sobel: magnitud combinada sqrt(dx^2 + dy^2)

# 2. Canny: detector òptim multietapa (Gaussià -> Gradient -> NMS -> Hysteresis)
# - Requereix imatge en escala de grisos
# - Retorna una màscara booleana (True = contorn)
if 'fruit_gray' in locals():
    edges_canny = canny(fruit_gray, sigma=1.5, low_threshold=0.1, high_threshold=0.2)
    print("Canny executat → contorns detectats:", np.sum(edges_canny), "píxels")


---
## 9. `skimage.segmentation` – Segmentació d'imatges [AMPLIAT P3]

La segmentació divideix una imatge en regions homogènies (superpíxels o segments) segons criteris de color, brillantor i continuïtat espacial. A la P3 s'analitzen i comparen en profunditat:
1. **Felzenszwalb**: basat en teoria de grafs (arbres d'expansió mínima).
2. **SLIC (Simple Linear Iterative Clustering)**: variant de K-Means en l'espai 5D $(L, a, b, x, y)$.
3. **Quickshift**: algorisme de cerca de modes (mean shift) local.
4. **`mark_boundaries`**: utilitat clau per visualitzar la partició superposada a la imatge.


In [ ]:
from skimage.segmentation import felzenszwalb, mark_boundaries

# ══════════════════════════════════════════════════════════════════
# 9.1 Felzenszwalb's Graph-Based Image Segmentation
# ══════════════════════════════════════════════════════════════════
# Modela la imatge com un graf on cada píxel és un node i les arestes representen diferència de color.
# Agrupa iterativament nodes minimitzant un criteri d'inconsistència interna.

# Paràmetres clau:
# - scale: controla el nivell de detall / mida dels components.
#          Valors alts (ex. 300-500) produeixen segments més grans; baixos (ex. 50-100) segments més fins.
# - sigma: desviació del kernel gaussià usat per pre-suavitzar la imatge abans de construir el graf.
#          Compensa artefactes de vores d'alta freqüència.
# - min_size: mida mínima (en píxels) d'un segment. Els components menors es fusionen amb el veí.

# 💡 No és obligatori convertir a grisos: Felzenszwalb aprofita directament els 3 canals de color RGB.
# ⚠️ Molt sensible al soroll (gaussià o salt-and-pepper), ja que qualsevol píxel anòmal altera les arestes del graf.

if 'img_rgb' in locals():
    segments_fz = felzenszwalb(img_rgb, scale=200, sigma=0.8, min_size=100)
    print("Felzenszwalb segments únics:", len(np.unique(segments_fz)))
    print("Shape de l'etiquetatge:", segments_fz.shape, "dtype:", segments_fz.dtype)


In [ ]:
from skimage.segmentation import slic

# ══════════════════════════════════════════════════════════════════
# 9.2 SLIC — Simple Linear Iterative Clustering
# ══════════════════════════════════════════════════════════════════
# Aplica K-Means en un espai 5D compost per les dimensions de color CIELAB [L, a, b]
# i les coordenades espacials [x, y]. Genera superpíxels uniformes i compactes.

# Paràmetres clau:
# - n_segments: nombre aproximat de superpíxels desitjats.
# - compactness: pondera la importància de la proximitat espacial respecte a la semblança de color.
#                Valors alts (ex. 20-40) forcen superpíxels més regulars/quadrats;
#                Valors baixos (ex. 1-10) s'adapten millor als contorns reals de color però són menys regulars.
# - sigma: suavitzat gaussià previ a cada canal.
# - start_label=1: fa que les etiquetes comencin a 1 en lloc de 0.

# 💡 Avantatge: molt més robust davant el soroll que Felzenszwalb, perquè el K-Means promedia sobre regions locals.

if 'img_rgb' in locals():
    segments_slic = slic(img_rgb, n_segments=200, compactness=15, sigma=1.0, start_label=1)
    print("SLIC superpíxels únics:", len(np.unique(segments_slic)))


In [ ]:
from skimage.segmentation import quickshift

# ══════════════════════════════════════════════════════════════════
# 9.3 Quickshift Image Segmentation
# ══════════════════════════════════════════════════════════════════
# Algorisme de cerca de modes no paramètric (variant de mean-shift).
# Cada píxel es connecta amb el veí més proper que tingui una densitat de probabilitat estimada superior.

# Paràmetres clau:
# - kernel_size: amplada de la finestra gaussiana en l'espai de posició/color. Controla l'escala espacial.
#                Valors més grans (ex. 7-10) resulten en menys segments i més grans.
# - max_dist: distància màxima entre punts en l'espai conjunt (espai + color) per unir-los en un mateix cluster.
# - ratio: equilibri entre la importància de la distància de color i la distància espacial (rang [0, 1]).

if 'img_rgb' in locals():
    segments_qs = quickshift(img_rgb, kernel_size=5, max_dist=10, ratio=0.5)
    print("Quickshift segments únics:", len(np.unique(segments_qs)))


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 9.4 mark_boundaries — Visualització de segments
# ══════════════════════════════════════════════════════════════════
# Superposa línies divisòries de color als marges dels segments sobre la imatge original.
# - color: tupla RGB de la vora (ex: (1, 1, 0) groc, (1, 0, 0) vermell).
# - mode: 'thick', 'inner', 'outer', 'subpixel'.
# - Retorna SEMPRE una imatge RGB float64 [0.0, 1.0].

if 'img_rgb' in locals():
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(mark_boundaries(img_rgb, segments_fz, color=(1, 1, 0)))
    ax[0].set_title(f"Felzenszwalb ({len(np.unique(segments_fz))} segments)")
    ax[0].axis("off")

    ax[1].imshow(mark_boundaries(img_rgb, segments_slic, color=(1, 1, 0)))
    ax[1].set_title(f"SLIC ({len(np.unique(segments_slic))} superpíxels)")
    ax[1].axis("off")

    ax[2].imshow(mark_boundaries(img_rgb, segments_qs, color=(1, 1, 0)))
    ax[2].set_title(f"Quickshift ({len(np.unique(segments_qs))} segments)")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()


---
## 10. `sklearn.cluster` & `sklearn.feature_extraction.image` – Clustering jeràrquic basat en grafs [NOU P3]

Aquest enfocament combina aprenentatge no supervisat amb restriccions espacials mitjançant la matriu d'adjacència de la graella de píxels:
1. `grid_to_graph(h, w)`: construeix un graf de connectivitat 2D (veïnatge de 4 píxels connectats) evitant que píxels allunyats es fusionin només pel color.
2. `AgglomerativeClustering`: agrupament jeràrquic ascendent amb criteri de variància mínima (`ward`).

⚠️ **Nota d'eficiència computacional**: La matriu de connectivitat i el càlcul de distàncies creixen ràpidament. Per imatges mitjanes/grans cal aplicar prèviament `rescale` o `resize` (downsampling), altrament es pot esgotar la memòria RAM.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph

# Exemple complet tal com es fa a la P3 (ex: Groot):
if 'fruit_gray' in locals():
    # 1. Reduïm la mida per fer el càlcul viable en memòria
    small_gray = rescale(fruit_gray, 0.2, anti_aliasing=True)
    h_s, w_s = small_gray.shape

    # 2. Generem la matriu de connectivitat espacial de la graella (graf 4-veïns)
    connectivity = grid_to_graph(h_s, w_s)

    # 3. Configurem el clustering jeràrquic amb connectivitat restringida
    clustering = AgglomerativeClustering(
        n_clusters=8, 
        linkage='ward', 
        connectivity=connectivity
    )

    # 4. Ajustem sobre els valors d'intensitat (reshape a vector columna N x 1)
    labels = clustering.fit_predict(small_gray.reshape(-1, 1))

    # 5. Reconstruïm la màscara 2D d'etiquetes
    segmented_img = labels.reshape(h_s, w_s)

    # 6. Visualització
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(small_gray, cmap='gray')
    ax[0].set_title(f"Imatge reduïda ({h_s}x{w_s})")
    ax[0].axis("off")

    ax[1].imshow(segmented_img, cmap='tab10')  # cmap='tab10' per distingir fàcilment clusters discrets
    ax[1].set_title("Agglomerative Clustering (n_clusters=8)")
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()


---
## 11. Subtracció de fons per a detecció de moviment (Background Subtraction) [NOU P3]

La subtracció de fons permet aïllar objectes en moviment respecte a un fons estàtic en una seqüència de vídeo o frames:

1. **Generació del model de fons**:
   - Es calcula la **mediana temporal** píxel a píxel d'una seqüència d'entrenament:
     $$B(x, y) = \text{median}_{t=1..N}(I_t(x, y))$$
   - La mediana és molt més robusta que la mitjana davant vianants o cotxes en moviment fugaç.
2. **Detecció de moviment**:
   - Es computa la diferència absoluta entre el frame de test i el model de fons:
     $$\Delta(x, y) = |I_{\text{test}}(x, y) - B(x, y)|$$
3. **Llindarització (Thresholding)**:
   - S'aplica un llindar $\tau$ per crear una màscara binària:
     $$M(x, y) = (\Delta(x, y) > \tau)$$


In [ ]:
# Codi patró de subtracció de fons tal com s'aplica a la P3:

# Pas 1: Carregar seqüència de frames d'entrenament (ex: 100 primers fotogrames)
# images_train = []
# for i in range(100):
#     img = io.imread(f"images_p3/vip_cup_2020/train/01_fisheye_day_{i:06d}.jpg")
#     img_res = resize(img, (img.shape[0] // 2, img.shape[1] // 2), anti_aliasing=True)
#     images_train.append(img_res)

# Pas 2: Computar el fons com a mediana temporal a l'eix axis=0
# bck_image = np.median(images_train, axis=0).astype(np.float64)

# Pas 3: Restar el model als fotogrames de test i aplicar llindar
# threshold = 0.1
# img_diff = np.abs(test_frame - bck_image)
# binary_mask = img_diff > threshold

# Exemple conceptual amb arrays simulats:
toy_frames = np.random.rand(20, 50, 50, 3)
toy_bck = np.median(toy_frames, axis=0)
toy_test = toy_frames[5] + 0.3 * (np.random.rand(50, 50, 3) > 0.95)
toy_diff = np.abs(toy_test - toy_bck)
toy_mask = np.mean(toy_diff, axis=-1) > 0.15

print("Background model shape:", toy_bck.shape, "| Binary mask shape:", toy_mask.shape)


---
## 12. Template Matching – Coincidència de patrons (SSD & NCC) [NOU P3]

El **Template Matching** busca la posició d'un subpatró o plantilla $T$ de mida $(h_T, w_T)$ dins una imatge font $I$ de mida $(H, W)$.

### 12.1 Mètode SSD (Sum of Squared Differences)
- Mesura de distància / error quadràtic:
  $$\text{SSD}(i, j) = \sum_{y, x} (I(i+y, j+x) - T(y, x))^2 = \| I_{i,j} - T \|_2^2$$
- Els **mínims** locals indiquen la màxima coincidència.
- Requereix llindaritzar per sota d'un valor: $\text{SSD} < \tau_{\text{ssd}}$.

### 12.2 Mètode NCC (Normalized Cross-Correlation)
- `skimage.feature.match_template`:
  Calcula el coeficient de correlació creuada normalitzat entre la plantilla i cada finestra de la imatge.
- Valors acotats entre $[-1.0, 1.0]$.
- Els **màxims** locals indiquen la màxima coincidència.
- Requereix llindaritzar per sobre d'un valor: $\text{NCC} > \tau_{\text{ncc}}$.

### 12.3 Coordenades i Bounding Boxes
- Com que la convolució / finestra lliscant només es pot aplicar on la plantilla cap completament, la matriu resultant té mida $(H - h_T + 1, W - w_T + 1)$.
- Per alinear visualment amb la imatge original es fa servir `np.pad(..., ((0, h_T - 1), (0, w_T - 1)), mode='edge')`.
- Les coordenades detectades s'extreuen amb `np.argwhere(mascara)` i es dibuixen amb `matplotlib.patches.Rectangle`.

### 12.4 Invariància a la Rotació
- **Importantíssim**: Ni SSD ni NCC són invariants a la rotació. Si la plantilla gira fins i tot pocs graus (ex. 5°-10° amb `skimage.transform.rotate`), la correlació cau dràsticament i apareixen falsos positius o pèrdua total de la detecció.


In [ ]:
from skimage.feature import match_template

def ssd_ncc_template_matching(img_source, img_template, threshold_ssd=8.2, threshold_ncc=0.6):
    """
    Aplica Template Matching usant SSD (càlcul manual) i NCC (skimage match_template).
    """
    h_img, w_img = img_source.shape
    h_tpl, w_tpl = img_template.shape

    # --- 1. SSD Valid (Sliding window manual) ---
    ssd_valid = np.zeros((h_img - h_tpl + 1, w_img - w_tpl + 1))
    for i in range(ssd_valid.shape[0]):
        for j in range(ssd_valid.shape[1]):
            sub_image = img_source[i : i + h_tpl, j : j + w_tpl]
            # Norma euclidiana de la diferència (o sum((sub - tpl)**2))
            ssd_valid[i, j] = np.linalg.norm(sub_image - img_template)

    # Padding per alinear la mida al pla d'imatge original
    ssd_map = np.pad(ssd_valid, ((0, h_tpl - 1), (0, w_tpl - 1)), mode='edge')
    thresholded_ssd = ssd_map < threshold_ssd

    # --- 2. NCC Valid (skimage.feature.match_template) ---
    ncc_valid = match_template(img_source, img_template)
    ncc_map = np.pad(ncc_valid, ((0, h_tpl - 1), (0, w_tpl - 1)), mode='edge')
    thresholded_ncc = ncc_map > threshold_ncc

    return ssd_map, thresholded_ssd, ncc_map, thresholded_ncc

# Funció per dibuixar les capses delimitadores (bounding boxes)
def add_bounding_boxes(axis, threshold_mask, h_tpl, w_tpl, step=20):
    peaks = np.argwhere(threshold_mask)
    for y, x in peaks[::step]:
        rect = patches.Rectangle((x, y), w_tpl, h_tpl, linewidth=1.5, edgecolor='r', facecolor='none')
        axis.add_patch(rect)

# Demostració si les imatges de prova existeixen:
if 'img_gray' in locals():
    try:
        tpl_eye = io.imread("images_p3/eye.png", as_gray=True)
        ssd_map, th_ssd, ncc_map, th_ncc = ssd_ncc_template_matching(
            img_gray, tpl_eye, threshold_ssd=8.2, threshold_ncc=0.6
        )
        print("Template matching calculat correctament:")
        print("- Coincidències SSD:", np.sum(th_ssd))
        print("- Coincidències NCC:", np.sum(th_ncc))
    except Exception as e:
        print("Nota:", e)


---
## 13. `matplotlib.pyplot` & `patches` – Visualització i anotació [AMPLIAT P3]


In [ ]:
# Mètodes clau de visualització utilitzats a les pràctiques:
# - plt.subplots(rows, cols, figsize=(W, H)): crea graella de subplots
# - ax.imshow(img, cmap=...):
#     - cmap='gray': per imatges monocromàtiques
#     - cmap='viridis': mapa continu de calor per a mapes SSD / NCC
#     - cmap='tab10' / 'nipy_spectral': mapa discret per a màscares de segments/clusters
# - ax.axis('off'): amaga els eixos de coordenades
# - patches.Rectangle((x, y), width, height, linewidth=..., edgecolor='r', facecolor='none'):
#     per dibuixar caixes delimitadores de deteccions
# - ax.add_patch(rect): afegeix la capsa al gràfic
# - fig.text(x, y, "Títol", ha='center', fontsize=...): col·loca títols globals o de bloc
# - plt.tight_layout(): reajusta marges automàticament


---
## 14. `numpy` – Operacions clau sobre arrays i mètriques [AMPLIAT P3]


In [ ]:
# Operacions de NumPy imprescindibles a P2 i P3:
# 1. np.median(arr, axis=0)     → calcula la mediana al llarg del temps/seqüència (fons estàtic)
# 2. np.abs(a - b)              → diferència absoluta píxel a píxel (motion detection)
# 3. np.linalg.norm(arr)        → norma euclidiana / distància d'un vector o bloc (SSD)
# 4. np.pad(arr, pad_width, mode='edge') → estén les vores d'una matriu
# 5. np.argwhere(condicio)      → retorna els índexs (fila, col) on es compleix una condició booleana
# 6. np.unique(arr)             → llista ordenada dels valors únics (comptar segments/etiquetes)
# 7. arr.reshape(-1, 1)         → transforma imatge 2D a matriu columna de mostres per a sklearn
# 8. arr.ravel()                → aplana l'array a 1D (per a càlcul d'histogrames)
# 9. np.zeros((H, W))           → matriu de zeros inicialitzadora (float64 per defecte)
# 10. np.zeros_like(img)        → matriu buida conservant forma i dtype exactes


---
## 📋 Taula resum de funcions (Pràctica 2 + Pràctica 3)

Aquesta taula detalla la biblioteca, mòdul, format d'entrada, tipus de retorn i observacions clau de cada funció utilitzada:

| Funció | Mòdul | Tipus d'entrada | Retorna (dtype · shape · rang) | Notes i Atenció |
|---|---|---|---|---|
| `io.imread(path, as_gray=...)` | `skimage.io` | Ruta a fitxer (str) | `uint8` · `(H,W,3)` o `(H,W)` · `[0, 255]` | Amb `as_gray=True` retorna `float64` `[0.0, 1.0]` |
| `rgb2gray(img)` | `skimage.color` | `(H,W,3)` qualsevol dtype | `float64` · `(H,W)` · `[0.0, 1.0]` | Conversió per lluminància ponderada |
| `img_as_float(img)` | `skimage` | qualsevol dtype | `float64` · mateixa shape · `[0.0, 1.0]` | Divideix uint8 entre 255 de forma segura |
| `img_as_ubyte(img)` | `skimage` | `float64` · `[0.0, 1.0]` | `uint8` · mateixa shape · `[0, 255]` | ⚠️ Error/clipping si hi ha valors fora de `[0, 1]` |
| `rescale(img, scale, ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · nova shape · `[0.0, 1.0]` | `preserve_range=True`, `anti_aliasing=True` |
| `resize(img, (H,W), ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · `(H,W)` o `(H,W,3)` · `[0.0, 1.0]` | Sempre retorna `float64` |
| `rotate(img, angle, ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · shape pot variar · `[0.0, 1.0]` | `resize=True` evita tallar les cantonades |
| `histogram(img, nbins)` | `skimage.exposure` | qualsevol | `(hist, bin_centers)` arrays 1D | Càlcul analític de freqüències per bin |
| `convolve(img, mask, mode)` | `scipy.ndimage` | qualsevol dtype | mateix dtype que entrada · mateixa shape | Si la màscara és float → sortida float |
| `gaussian(img, sigma)` | `skimage.filters` | qualsevol dtype | `float64` · mateixa shape · `[0.0, 1.0]` | Suavitzat continu; `channel_axis=-1` en RGB |
| `median(img, footprint)` | `skimage.filters` | qualsevol dtype (millor `uint8`) | **mateix dtype que entrada** · mateixa shape | Excel·lent per a soroll impulsiu (salt & pepper) |
| `disk(r)`, `rectangle`, `diamond` | `skimage.morphology` | Mides enteres | `uint8` de valors binòmis `{0, 1}` | Defineix el veïnatge (*footprint*) del filtre |
| `random_noise(img, mode)` | `skimage.util` | qualsevol | `float64` · mateixa shape · `[0.0, 1.0]` | Modes `'gaussian'`, `'s&p'`, etc. Sempre retorna float64 |
| `sobel(img)`, `prewitt(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape | Magnitud total del gradient espacial |
| `sobel_h`, `sobel_v` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape | Gradients direccionals parcials X i Y |
| `canny(img, sigma, ...)` | `skimage.feature` | `uint8` o `float64` grisos | `bool` · mateixa shape · `{True, False}` | Detector multi-etapa amb hysteresis |
| `felzenszwalb(img, ...)` | `skimage.segmentation` | RGB o grisos | `int32` · `(H,W)` · etiquetes de segment | Basat en grafs; sensible al soroll; no cal grisos |
| `slic(img, ...)` | `skimage.segmentation` | RGB qualsevol dtype | `int32` · `(H,W)` · etiquetes de superpíxels | Basat en K-Means 5D; robust al soroll; `start_label=1` |
| `quickshift(img, ...)` | `skimage.segmentation` | RGB qualsevol dtype | `int32` · `(H,W)` · etiquetes de segment | Cerca de modes local; paràmetres `kernel_size`, `max_dist` |
| `mark_boundaries(img, seg)` | `skimage.segmentation` | img qualsevol + seg int | `float64` · `(H,W,3)` · `[0.0, 1.0]` | Retorna sempre imatge RGB amb les línies superposades |
| `grid_to_graph(h, w)` | `sklearn.feature_extraction.image` | Dims enteres `h, w` | Matriu dispersa (`scipy.sparse.coo_matrix`) | Graf de connectivitat 4-veïns de la graella de píxels |
| `AgglomerativeClustering(...)` | `sklearn.cluster` | Matriu de dades `(N, 1)` | Objecte clusterer (`labels_` o `.fit_predict()`) | Clustering jeràrquic restringit pel graf de connectivitat |
| `match_template(img, tpl)` | `skimage.feature` | `float64` en escala de grisos | `float64` · `(H-h+1, W-w+1)` · `[-1.0, 1.0]` | Normalized Cross-Correlation (NCC); màxims = coincidència |
| `np.median(seq, axis=0)` | `numpy` | Llista o stack de frames 3D/4D | `float64` · mateixes dimensions espacials | Model de fons estàtic temporal |
| `np.abs(test - bck)` | `numpy` | Dos arrays de mateixa shape | Mateixa shape i dtype | Diferència de fotogrames per a detecció de moviment |
| `np.pad(arr, pad_width, ...)` | `numpy` | Array de qualsevol dimensió | Array ampliat amb padding | Alinia mapes de correlació a la mida original d'imatge |
| `np.linalg.norm(a - b)` | `numpy.linalg` | Vectors o matrius | Escalar float64 | Càlcul de distància euclidiana (SSD) |
| `np.argwhere(mask)` | `numpy` | Array booleà | Array 2D d'índexs `(N, d)` | Obté les coordenades de pics/deteccions |
| `np.unique(arr)` | `numpy` | Array de valors discrets | Array 1D ordenat de valors únics | Recompte de segments o etiquetes |
| `patches.Rectangle((x,y),w,h)` | `matplotlib.patches` | Coordenades i mides | Objecte gràfic `Rectangle` | Caixa delimitadora per visualitzar deteccions |
| `plt.imshow(img, cmap=...)` | `matplotlib.pyplot` | Array 2D o 3D | Objecte `AxesImage` | `cmap='gray'`, `'viridis'`, `'tab10'`, etc. |
| `plt.subplots(r, c, ...)` | `matplotlib.pyplot` | Nombre de files i columnes | `(Figure, array of Axes)` | Creació de figures multipanell |
